GOOGLE COLAB VSCODE EXTENSION SETUP

In [2]:
# Using Google Colab GPUs and TPUs with VSCode extension
import sys
if 'google.colab' in sys.modules:
    print('✅I am in Google Colab')
    print(f'Python version: {sys.version}')
else:
    print('❌Not in Google Colab')

✅I am in Google Colab
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
# Check Hardware specifications
import psutil
import os

# RAM
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"💾 RAM total: {ram_gb:.2f} GB")

# CPU
cpu_count = psutil.cpu_count()
print(f"⚙️ CPUs disponibles: {cpu_count}")

# Verificar GPU
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
if gpu_info:
    print(f"🎮 GPU: {gpu_info[0]}")
else:
    print("⚠️ No hay GPU disponible (usando CPU)")

🎮 GPU: Tesla T4, 15360 MiB


In [5]:
# Check working directory
from pathlib import Path
Path.cwd()

PosixPath('/content')

In [6]:
# Check if requirements.txt has been written into the folder
!ls

sample_data


In [8]:
# git clone the repo
!git clone https://github.com/llucliment/Twitter-disaster.git

Cloning into 'Twitter-disaster'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 52 (delta 20), reused 50 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 614.75 KiB | 2.87 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [9]:
# Move into the repo
%cd /content/Twitter-disaster

/content/Twitter-disaster


In [10]:
# Bring latest changes to the Colab repo
!git pull

Already up to date.


In [11]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 98.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 134.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: jupyter-server
    Found existing installation: jupyter_server 2.18.2
    Uninstalling jupyter_server-2.18.2:
      Successfully uninstalled jupyter_server-2.18.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires jupyter-server==2.18.2, but you have jupy

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from transformers import AutoModelForSequenceClassification
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
import copy

from src.dataset import TweetDataset
from src.utils import seed_everything


In [13]:
df = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [14]:
print(torch.__version__)

2.11.0+cu128


In [15]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

seed_everything(42)
device

device(type='cuda')

In [16]:
import re

def preprocess_bertweet(text):
    """ Preprocessing taking into account Bertweet training """
    text = str(text)

    # Normalize URLs to BERTweet's expected token
    text = re.sub(r"https?://\S+|www\.\S+", "HTTPURL", text)

    # Normalize user mentions to BERTweet's expected token
    text = re.sub(r"@\w+", "@USER", text)

    # Optional: normalize extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [17]:
df["input_text"] = (df["keyword"].fillna("").str.replace("%20", " ", regex=False)
                    + " "
                    + df["text"].fillna("").apply(preprocess_bertweet))

test["input_text"] = (test["keyword"].fillna("").str.replace("%20", " ", regex=False)
                      + " "
                      + test["text"].fillna("").apply(preprocess_bertweet))

In [18]:
df

,id,keyword,location,text,target,input_text
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1,Our Deeds are the Reason of this #earthquake ...
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1,Forest fire near La Ronge Sask. Canada
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1,All residents asked to 'shelter in place' are...
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1,"13,000 people receive #wildfires evacuation o..."
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1,Just got sent this photo from Ruby #Alaska as...
...,...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1,Two giant cranes holding a bridge collapse in...
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1,@USER @USER The out of control wild fires in ...
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. HT...
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1,Police investigating after an e-bike collided...


In [19]:
# TRAIN TEST SPLIT 
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

len(train_df), len(val_df)

(6090, 1523)

LOAD BERTWEET TOKENIZER

In [ ]:
from transformers import AutoTokenizer
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

MODEL_NAME = "vinai/bertweet-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


CREATE DATALOADERS

In [10]:
from torch.utils.data import DataLoader

MAX_LENGTH = 128
BATCH_SIZE = 16

train_dataset = TweetDataset(
    texts=train_df["text"],
    labels=train_df["target"].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

val_dataset = TweetDataset(
    texts=val_df["text"],
    labels=val_df["target"].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = TweetDataset(
    texts=test["text"],
    labels=None,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

LOAD BERTWEET FOR CLASIFICATION

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


MODEL

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

def train_one_epoch_bert(model, loader, optimizer, scheduler=None):
    model.train()

    total_loss = 0

    for batch in tqdm(loader, desc="Training"):
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * batch["input_ids"].size(0)

    return total_loss / len(loader.dataset)

In [14]:
def evaluate_bert(model, loader, threshold=0.5):
    model.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            outputs = model(**batch)

            loss = outputs.loss
            logits = outputs.logits

            probs = torch.softmax(logits, dim=1)[:, 1]

            total_loss += loss.item() * batch["input_ids"].size(0)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    preds = (all_probs >= threshold).astype(int)
    f1 = f1_score(all_labels, preds)

    return total_loss / len(loader.dataset), f1, all_probs, all_labels

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

TRAIN

In [15]:
# EARLY STOPPING 
num_epochs = 100
patience = 2
min_delta = 0.0001

best_f1 = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(num_epochs):
    train_loss = train_one_epoch_bert(
        model,
        train_loader,
        optimizer
    )

    val_loss, val_f1, val_probs, val_labels = evaluate_bert(
        model,
        val_loader,
        threshold=0.5
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1={val_f1:.4f}"
    )

    if val_f1 > best_f1 + min_delta:
        best_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        print("New best model saved.")
    else:
        epochs_without_improvement += 1

        print(
            f"No improvement for "
            f"{epochs_without_improvement}/{patience} epochs."
        )

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)

print(f"Best validation F1: {best_f1:.4f}")

Training:   0%|          | 0/381 [00:00<?, ?it/s]

KeyboardInterrupt: 